# 位置符号化（Positional Encoding）

このノートブックでは、Transformer の **位置符号化（Positional Encoding）** を学びます。

前回の Multi-Head Attention では、トークン同士の関連度（内積）を計算しましたが、
**トークンの順番の情報が欠けている** という致命的な欠点がありました。

位置符号化はこの問題を解決する仕組みです。

## 目次
1. なぜ位置情報が必要なのか？
2. 位置符号化の数式（式4-3）
3. 具体的な計算例（図4.30）
4. 位置符号化行列の全体像
5. 埋め込みベクトルへの加算（図4.31）
6. sin / cos を使う理由
7. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# エンコーダのどの部分を学んでいるか？
fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(-1, 15)
ax.set_ylim(-0.5, 2.5)
ax.axis('off')

blocks = [
    ('Input\nEmbedding', '#9E9E9E', '#F5F5F5', False),
    ('Positional\nEncoding', '#E65100', '#FFF3E0', True),   # ★ 今ここ！
    ('Multi-Head\nAttention', '#9E9E9E', '#F5F5F5', False),
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
    ('Feed\nForward', '#9E9E9E', '#F5F5F5', False),
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
]

for i, (label, edge_color, face_color, highlight) in enumerate(blocks):
    x = i * 2.3
    lw = 3 if highlight else 1
    rect = mpatches.FancyBboxPatch((x, 0.3), 1.8, 1.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor=face_color, edgecolor=edge_color, linewidth=lw)
    ax.add_patch(rect)
    ax.text(x + 0.9, 1.05, label, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(blocks) - 1:
        ax.annotate('', xy=(x + 2.1, 1.05), xytext=(x + 1.85, 1.05),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.text(2.3 + 0.9, 2.2, '\u2190 \u4eca\u3053\u3053\uff01', fontsize=12, fontweight='bold', color='#E65100', ha='center')
ax.set_title('\u30a8\u30f3\u30b3\u30fc\u30c0\u306e\u69cb\u6210 \u2014 Positional Encoding \u3092\u5b66\u3076', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1. なぜ位置情報が必要なのか？

前回のノートブックで学んだ Q₁K₁ᵀ は、各トークン同士の **関連度** を計算できます。

しかし、1つ **致命的な欠点** があります。

### 語順が変わると意味が変わる

自然言語は、**単語の並び方によって文意が決まります**。
単語の順番が変わると、文章の意味そのものが変わってしまう可能性があります。

| 文章 | 意味 |
|------|------|
| 「**彼は犬に**追われていた」 | 彼が追われている |
| 「**犬は彼に**追われていた」 | 犬が追われている |

同じ単語を使っているのに、順番を変えるだけで **全く逆の意味** になります。

### 単語埋め込みだけでは語順がわからない

単語埋め込み（Word Embedding）は、各トークンを独立にベクトルに変換します。
「Mount」が1番目にあっても7番目にあっても、同じベクトルが生成されます。

**→ 位置の情報が失われている！**

そこで、各トークンに **「あなたは何番目ですよ」という位置の情報を付与** するのが
**位置符号化（Positional Encoding）** です。

In [ ]:
# 語順が変わると意味が変わることを可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 文1
ax = axes[0]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
words1 = ['\u5f7c\u306f', '\u72ac\u306b', '\u8ffd\u308f\u308c\u3066\u3044\u305f']
for i, w in enumerate(words1):
    color = '#E3F2FD' if i < 2 else '#FFF9C4'
    ax.text(1 + i * 3, 2.5, w, ha='center', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color, edgecolor='gray'))
ax.text(5, 1, '\u2192 \u5f7c\u304c\u8ffd\u308f\u308c\u3066\u3044\u308b', ha='center', fontsize=13, color='#1565C0', fontweight='bold')
ax.set_title('\u6587\u7ae01', fontsize=12)

# 文2（語順を変えた）
ax = axes[1]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
words2 = ['\u72ac\u306f', '\u5f7c\u306b', '\u8ffd\u308f\u308c\u3066\u3044\u305f']
for i, w in enumerate(words2):
    color = '#FFCDD2' if i < 2 else '#FFF9C4'
    ax.text(1 + i * 3, 2.5, w, ha='center', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color, edgecolor='gray'))
ax.text(5, 1, '\u2192 \u72ac\u304c\u8ffd\u308f\u308c\u3066\u3044\u308b', ha='center', fontsize=13, color='#C62828', fontweight='bold')
ax.set_title('\u6587\u7ae02\uff08\u8a9e\u9806\u3092\u5909\u3048\u305f\uff09', fontsize=12)

plt.suptitle('\u540c\u3058\u5358\u8a9e\u3067\u3082\u8a9e\u9806\u304c\u5909\u308f\u308b\u3068\u610f\u5473\u304c\u5168\u304f\u9055\u3046\uff01', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\u2192 \u5358\u8a9e\u540c\u58eb\u306e\u95a2\u9023\u6027\u3060\u3051\u3067\u306f\u3001\u3053\u306e\u8a9e\u9806\u306e\u554f\u984c\u3092\u89e3\u6c7a\u3067\u304d\u306a\u3044")
print("\u2192 \u305d\u3053\u3067\u3001\u5404\u30c8\u30fc\u30af\u30f3\u306b\u300c\u4f4d\u7f6e\u306e\u60c5\u5831\u300d\u3092\u4ed8\u4e0e\u3059\u308b\u306e\u304c Positional Encoding")

## 2. 位置符号化の数式（式4-3）

Transformer の原論文「Attention Is All You Need」では、
**sin / cos 関数** を使った位置符号化が提案されています。

### 式(4-3)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)$$

### 各変数の意味

| 変数 | 意味 | 本書のケース |
|------|------|-------------|
| $pos$ | トークンの位置（1番目なら pos=1）| 1〜7 |
| $i$ | 次元のインデックス（0から始まる）| 0, 1, 2 |
| $d_{model}$ | トークンのベクトルの次元数 | 6 |
| $2i$ | 偶数番目の次元 → sin を使う | 0, 2, 4 |
| $2i+1$ | 奇数番目の次元 → cos を使う | 1, 3, 5 |

### ポイント

- **偶数番目の次元には sin**、**奇数番目の次元には cos** を使う
- pos と i の組み合わせごとに **すべて異なる値** が生成される
- これにより各トークンの各次元に **ユニークな位置情報** が付与される

In [ ]:
# 式(4-3) を Python で実装

def positional_encoding(pos, d_model):
    """
    1つのトークン位置 pos に対する位置符号化ベクトルを計算する
    
    Parameters:
        pos: トークンの位置（1始まり）
        d_model: ベクトルの次元数
    Returns:
        PE ベクトル（d_model 次元）
    """
    PE = np.zeros(d_model)
    for dim in range(d_model):
        i = dim // 2  # i の値
        angle = pos / (10000 ** (2 * i / d_model))
        if dim % 2 == 0:  # 偶数番目 → sin
            PE[dim] = np.sin(angle)
        else:             # 奇数番目 → cos
            PE[dim] = np.cos(angle)
    return PE

d_model = 6
print("=== \u5f0f(4-3) \u306e\u5b9f\u88c5 ===")
print(f"d_model = {d_model}")
print()
print("\u5404\u6b21\u5143\u306e\u8a08\u7b97\u5f0f:")
print("  \u6b21\u51430 (2i=0,   i=0): PE(pos,0) = sin(pos / 10000^(0/6))")
print("  \u6b21\u51431 (2i+1=1, i=0): PE(pos,1) = cos(pos / 10000^(0/6))")
print("  \u6b21\u51432 (2i=2,   i=1): PE(pos,2) = sin(pos / 10000^(2/6))")
print("  \u6b21\u51433 (2i+1=3, i=1): PE(pos,3) = cos(pos / 10000^(2/6))")
print("  \u6b21\u51434 (2i=4,   i=2): PE(pos,4) = sin(pos / 10000^(4/6))")
print("  \u6b21\u51435 (2i+1=5, i=2): PE(pos,5) = cos(pos / 10000^(4/6))")

## 3. 具体的な計算例（図4.30）

pos=1（1番目のトークン「Mount」）、d_model=6 のケースで具体的に計算してみましょう。

i = 0, 1, 2 を順に代入していきます。

In [ ]:
# 図4.30: pos=1, d_model=6 での計算過程を詳しく見る

pos = 1
d_model = 6

print(f"=== \u56f34.30: pos={pos}, d_model={d_model} \u306e\u8a08\u7b97 ===")
print()

for i in range(d_model // 2):  # i = 0, 1, 2
    dim_even = 2 * i      # 偶数次元
    dim_odd = 2 * i + 1   # 奇数次元
    
    exponent = 2 * i / d_model
    denominator = 10000 ** exponent
    angle = pos / denominator
    
    pe_sin = np.sin(angle)
    pe_cos = np.cos(angle)
    
    print(f"--- i = {i} ---")
    print(f"  10000^(2\u00d7{i}/{d_model}) = 10000^{exponent:.4f} = {denominator:.4f}")
    print(f"  \u89d2\u5ea6 = pos / {denominator:.4f} = {pos} / {denominator:.4f} = {angle:.6f}")
    print(f"  PE({pos},{dim_even}) = sin({angle:.6f}) = {pe_sin:.6f}  \u2190 \u6b21\u5143{dim_even}\uff08\u5076\u6570\uff09")
    print(f"  PE({pos},{dim_odd}) = cos({angle:.6f}) = {pe_cos:.6f}  \u2190 \u6b21\u5143{dim_odd}\uff08\u5947\u6570\uff09")
    print()

# 結果まとめ
pe_1 = positional_encoding(1, d_model)
print(f"\u2605 pos=1 \u306e\u4f4d\u7f6e\u7b26\u53f7\u5316\u30d9\u30af\u30c8\u30eb:")
print(f"  PE(1) = [{', '.join(f'{v:.6f}' for v in pe_1)}]")

In [ ]:
# 全7トークンの位置符号化を計算

n_tokens = 7
d_model = 6
words = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

# 位置符号化行列 (7×6)
PE_matrix = np.zeros((n_tokens, d_model))
for pos in range(1, n_tokens + 1):
    PE_matrix[pos - 1] = positional_encoding(pos, d_model)

print("=== \u5168\u30c8\u30fc\u30af\u30f3\u306e\u4f4d\u7f6e\u7b26\u53f7\u5316\u884c\u5217 (7\u00d76) ===")
print()
header = f"{'pos':>3s} {'\u30c8\u30fc\u30af\u30f3':12s}" + "".join(f"{'PE(,'+str(d)+')':>11s}" for d in range(d_model))
print(header)
func_row = f"{'':3s} {'':12s}" + "".join(f"{'sin' if d%2==0 else 'cos':>11s}" for d in range(d_model))
print(func_row)
print("-" * len(header))
for pos in range(n_tokens):
    vals = "".join(f"{v:11.6f}" for v in PE_matrix[pos])
    print(f"{pos+1:3d} {words[pos]:12s}{vals}")

print()
print("\u2605 \u5404\u30c8\u30fc\u30af\u30f3\u306e\u5404\u6b21\u5143\u306b\u7570\u306a\u308b\u5024\u304c\u5272\u308a\u5f53\u3066\u3089\u308c\u3066\u3044\u308b")
print("\u2605 \u3053\u308c\u304c\u300c\u4f4d\u7f6e\u306e\u60c5\u5831\u300d\u3068\u3057\u3066\u57cb\u3081\u8fbc\u307f\u30d9\u30af\u30c8\u30eb\u306b\u52a0\u7b97\u3055\u308c\u308b")

## 4. 位置符号化行列の全体像

位置符号化行列をヒートマップで可視化して、パターンを観察しましょう。

In [ ]:
# 位置符号化行列のヒートマップ

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左: 7×6 の位置符号化行列
ax = axes[0]
im = ax.imshow(PE_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels([f'pos={i+1} ({w})' for i, w in enumerate(words)], fontsize=9)
ax.set_xticks(range(d_model))
ax.set_xticklabels([f'd{i}\n({"sin" if i%2==0 else "cos"})' for i in range(d_model)], fontsize=8)
ax.set_title('\u4f4d\u7f6e\u7b26\u53f7\u5316\u884c\u5217 PE (7\u00d76)', fontsize=13, fontweight='bold')
for i in range(n_tokens):
    for j in range(d_model):
        color = 'white' if abs(PE_matrix[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{PE_matrix[i,j]:.3f}', ha='center', va='center', fontsize=7, color=color)
plt.colorbar(im, ax=ax)

# 右: より大きな行列で波形パターンを見る（50トークン×64次元）
ax = axes[1]
d_large = 64
n_large = 50
PE_large = np.zeros((n_large, d_large))
for pos in range(1, n_large + 1):
    PE_large[pos - 1] = positional_encoding(pos, d_large)

im2 = ax.imshow(PE_large, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('\u6b21\u5143', fontsize=11)
ax.set_ylabel('\u30c8\u30fc\u30af\u30f3\u4f4d\u7f6e (pos)', fontsize=11)
ax.set_title('\u4f4d\u7f6e\u7b26\u53f7\u5316\u306e\u6ce2\u5f62\u30d1\u30bf\u30fc\u30f3 (50\u00d764)', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=ax)

plt.suptitle('\u4f4d\u7f6e\u7b26\u53f7\u5316\u306e\u53ef\u8996\u5316', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\u30dd\u30a4\u30f3\u30c8:")
print("  \u5de6: \u66f8\u7c4d\u306e\u4f8b (7\u00d76) \u2014 \u5404\u30de\u30b9\u306b\u7570\u306a\u308b\u5024")
print("  \u53f3: \u3088\u308a\u5927\u304d\u306a\u884c\u5217 (50\u00d764) \u2014 sin/cos \u306e\u6ce2\u5f62\u30d1\u30bf\u30fc\u30f3\u304c\u898b\u3048\u308b")
print("  \u4f4e\u3044\u6b21\u5143\u307b\u3069\u5468\u671f\u304c\u77ed\u304f\u3001\u9ad8\u3044\u6b21\u5143\u307b\u3069\u5468\u671f\u304c\u9577\u3044")

## 5. 埋め込みベクトルへの加算（図4.31）

位置符号化は、単語埋め込みの結果に **加算** されます。

```
文字列: "Mount Fuji looks beautiful in spring."
  ↓ Tokenization（トークン化）
  ↓ Word Embedding（単語埋め込み）
埋め込み行列 (7×6)
  ＋  ← ここで加算！
位置符号化行列 (7×6)
  ＝
最終入力行列 (7×6)  →  Multi-Head Attention へ
```

**掛け算ではなく足し算** であることがポイントです。
要素ごとに位置の情報が「上乗せ」されます。

In [ ]:
# 図4.31: 埋め込みベクトル + 位置符号化 の可視化

np.random.seed(123)

# 単語埋め込み行列（前回のノートブックの出力）
embedding = np.round(np.random.randn(n_tokens, d_model) * 0.5, 3)

# 最終入力 = 埋め込み + 位置符号化
final_input = embedding + PE_matrix

fig, axes = plt.subplots(1, 5, figsize=(20, 5),
                         gridspec_kw={'width_ratios': [2, 0.3, 2, 0.3, 2]})

# 埋め込み行列
ax = axes[0]
im1 = ax.imshow(embedding, cmap='RdBu_r', aspect='auto', vmin=-1.5, vmax=1.5)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xticks(range(d_model))
ax.set_xticklabels([f'd{i}' for i in range(d_model)], fontsize=8)
ax.set_title('Word Embedding\n(7\u00d76)', fontsize=11, fontweight='bold', color='#1565C0')
for i in range(n_tokens):
    for j in range(d_model):
        color = 'white' if abs(embedding[i,j]) > 0.7 else 'black'
        ax.text(j, i, f'{embedding[i,j]:.2f}', ha='center', va='center', fontsize=6, color=color)

# + 記号
ax = axes[1]
ax.axis('off')
ax.text(0.5, 0.5, '+', ha='center', va='center', fontsize=30, fontweight='bold', color='#E65100')

# 位置符号化
ax = axes[2]
im2 = ax.imshow(PE_matrix, cmap='RdBu_r', aspect='auto', vmin=-1.5, vmax=1.5)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels([f'pos={i+1}' for i in range(n_tokens)], fontsize=9)
ax.set_xticks(range(d_model))
ax.set_xticklabels([f'd{i}' for i in range(d_model)], fontsize=8)
ax.set_title('Positional Encoding\n(7\u00d76)', fontsize=11, fontweight='bold', color='#E65100')
for i in range(n_tokens):
    for j in range(d_model):
        color = 'white' if abs(PE_matrix[i,j]) > 0.7 else 'black'
        ax.text(j, i, f'{PE_matrix[i,j]:.2f}', ha='center', va='center', fontsize=6, color=color)

# = 記号
ax = axes[3]
ax.axis('off')
ax.text(0.5, 0.5, '=', ha='center', va='center', fontsize=30, fontweight='bold', color='#2E7D32')

# 最終入力
ax = axes[4]
im3 = ax.imshow(final_input, cmap='RdBu_r', aspect='auto', vmin=-1.5, vmax=1.5)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xticks(range(d_model))
ax.set_xticklabels([f'd{i}' for i in range(d_model)], fontsize=8)
ax.set_title('\u6700\u7d42\u5165\u529b\n(7\u00d76)', fontsize=11, fontweight='bold', color='#2E7D32')
for i in range(n_tokens):
    for j in range(d_model):
        color = 'white' if abs(final_input[i,j]) > 0.7 else 'black'
        ax.text(j, i, f'{final_input[i,j]:.2f}', ha='center', va='center', fontsize=6, color=color)

plt.suptitle('\u56f34.31: Word Embedding + Positional Encoding = Multi-Head Attention \u3078\u306e\u5165\u529b',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\u57cb\u3081\u8fbc\u307f\u884c\u5217\u306e\u5f62\u72b6:   {embedding.shape}")
print(f"\u4f4d\u7f6e\u7b26\u53f7\u5316\u306e\u5f62\u72b6:   {PE_matrix.shape}")
print(f"\u6700\u7d42\u5165\u529b\u306e\u5f62\u72b6:     {final_input.shape}")
print()
print("\u2605 \u5f62\u72b6\u306f\u3059\u3079\u3066 7\u00d76 \u3067\u540c\u3058")
print("\u2605 \u8981\u7d20\u3054\u3068\u306b\u8db3\u3057\u7b97\u3059\u308b\u3060\u3051")
print("\u2605 \u3053\u306e\u6700\u7d42\u5165\u529b\u304c Multi-Head Attention \u306b\u6e21\u3055\u308c\u308b")

In [ ]:
# 1つのトークンで加算の過程を詳しく見る

token_idx = 0  # "Mount" (pos=1)
print(f'=== "{words[token_idx]}" (pos={token_idx+1}) \u306e\u52a0\u7b97\u904e\u7a0b ===')
print()

print(f"{'\u6b21\u5143':>6s}  {'\u57cb\u3081\u8fbc\u307f':>10s}  {'PE':>10s}  {'\u6700\u7d42\u5165\u529b':>10s}  {'\u95a2\u6570':>6s}")
print("-" * 55)
for d in range(d_model):
    func = 'sin' if d % 2 == 0 else 'cos'
    print(f"  d{d:1d}    {embedding[token_idx, d]:10.4f}  {PE_matrix[token_idx, d]:10.4f}  {final_input[token_idx, d]:10.4f}  {func:>6s}")

print()
print("\u2192 \u5404\u6b21\u5143\u306b\u4f4d\u7f6e\u56fa\u6709\u306e\u5024\u304c\u52a0\u7b97\u3055\u308c\u308b")
print("\u2192 \u540c\u3058\u5358\u8a9e\u3067\u3082\u4f4d\u7f6e\u304c\u9055\u3048\u3070\u7570\u306a\u308b\u30d9\u30af\u30c8\u30eb\u306b\u306a\u308b")

## 6. sin / cos を使う理由

なぜ位置情報に sin / cos を使うのでしょうか？他にも方法はありそうです。

### 他の方法との比較

| 方法 | 例 | 問題点 |
|------|-----|--------|
| 連番を使う | pos=1→1, pos=2→2, ... | 値の大きさがバラバラ。長い文で値が爆発する |
| 0〜1に正規化 | pos=1→0.14, pos=2→0.29, ... | 文の長さによって同じ位置でも値が変わる |
| 学習させる | 学習で最適な値を見つける | 学習データにない長さの文に対応できない |
| **sin / cos** | **周期関数を使う** | **上記の問題をすべて解決** |

### sin / cos の利点

1. **値の範囲が -1〜1 に収まる** → 値が爆発しない
2. **文の長さに依存しない** → 同じ位置なら常に同じ値
3. **学習不要** → 固定値なのでパラメータが増えない
4. **相対的な位置関係を捉えられる** → PE(pos+k) は PE(pos) の線形変換で表現可能

In [ ]:
# sin / cos の波形を可視化

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

positions = np.arange(1, 51)

for idx, i in enumerate([0, 1, 2]):
    ax = axes[idx]
    
    # sin (偶数次元)
    dim_even = 2 * i
    sin_vals = [positional_encoding(p, d_model)[dim_even] for p in positions]
    ax.plot(positions, sin_vals, 'b-', linewidth=2, label=f'd{dim_even} (sin, i={i})')
    
    # cos (奇数次元)
    dim_odd = 2 * i + 1
    cos_vals = [positional_encoding(p, d_model)[dim_odd] for p in positions]
    ax.plot(positions, cos_vals, 'r--', linewidth=2, label=f'd{dim_odd} (cos, i={i})')
    
    ax.set_ylabel(f'i={i}', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(-1.2, 1.2)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5)

axes[-1].set_xlabel('\u30c8\u30fc\u30af\u30f3\u4f4d\u7f6e (pos)', fontsize=12)
plt.suptitle('\u5404\u6b21\u5143\u306e sin/cos \u6ce2\u5f62\uff08i \u304c\u5927\u304d\u3044\u307b\u3069\u5468\u671f\u304c\u9577\u3044\uff09', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\u30dd\u30a4\u30f3\u30c8:")
print("  i=0: \u5468\u671f\u304c\u77ed\u3044 \u2192 \u8fd1\u3044\u4f4d\u7f6e\u306e\u9055\u3044\u3092\u6355\u3048\u308b")
print("  i=1: \u5468\u671f\u304c\u4e2d\u7a0b\u5ea6")
print("  i=2: \u5468\u671f\u304c\u9577\u3044 \u2192 \u9060\u3044\u4f4d\u7f6e\u306e\u9055\u3044\u3092\u6355\u3048\u308b")
print()
print("\u2192 \u8907\u6570\u306e\u5468\u671f\u3092\u7d44\u307f\u5408\u308f\u305b\u308b\u3053\u3068\u3067\u3001\u5404\u4f4d\u7f6e\u306b\u30e6\u30cb\u30fc\u30af\u306a\u30d1\u30bf\u30fc\u30f3\u3092\u5272\u308a\u5f53\u3066\u308b")

## 7. まとめ

| ポイント | 内容 |
|----------|------|
| **なぜ必要か** | 単語埋め込みだけではトークンの順番がわからない。語順で意味が変わるため必須 |
| **式(4-3)** | 偶数次元: $PE_{(pos,2i)} = \sin(pos / 10000^{2i/d_{model}})$、奇数次元: cos を使用 |
| **pos** | トークンの位置（1番目=1, 2番目=2, ...）|
| **i** | 次元のインデックス（i=0,1,2 → 次元0〜5）|
| **値の範囲** | -1〜1（sin/cos の性質）|
| **加算** | 埋め込みベクトルに要素ごとに足す（掛け算ではない）|
| **学習不要** | 固定値。パラメータが増えない |
| **波形パターン** | 低い次元は短い周期、高い次元は長い周期 |

### Transformer のここまでの処理の全体像

```
入力文: "Mount Fuji looks beautiful in spring."
  ↓ Tokenization（トークン化）
["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]
  ↓ Word Embedding（単語埋め込み）
7×6 行列
  ↓ + Positional Encoding（位置符号化を加算）  ← 今回学んだ
7×6 行列（位置情報付き）
  ↓ Multi-Head Attention（前回学んだ）
7×6 行列
  ↓ Add & Norm → Feed Forward → Add & Norm
  ↓ × N回繰り返し (N=6)
エンコーダの出力 (7×6)
```

## 次のステップ

次のノートブックでは、Multi-Head Attention の後に続く
**Add & Norm（残差接続と層正規化）** および **Feed Forward Network** を学びます。